# 10: Export experiment evidence for sharing

Run on a **CPU runtime**. This notebook creates one ZIP on Drive. It includes every Git-tracked repository file, all saved non-checkpoint files found under `E08_forgetting_trace`, a readable index, request-by-request timelines, and a manifest with a SHA-256 hash for each copied source file. That covers the archived experiment tables, per-step traces, original saved JSON, diagnostics, charts, notebooks, and code that are present when you run it.

Large `.pt` checkpoints are listed but excluded by default. Set `INCLUDE_CHECKPOINTS = True` if the recipient needs the model weights and you have enough Drive space. Files that were never saved or were lost in an old Colab session cannot be recovered by this export.

## SETUP-01: Use CPU, load the repository, and mount Drive


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import runpy
import torch
from google.colab import drive

if torch.cuda.is_available():
    raise RuntimeError("Choose a CPU runtime before creating the export.")
urlretrieve(
    "https://raw.githubusercontent.com/sumitasthana/CARK/main/scripts/colab_setup.py",
    "/content/uncle_colab_setup.py",
)
setup = runpy.run_path("/content/uncle_colab_setup.py", run_name="export_setup")
REPO = setup["setup_repo"](install=False)
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace")
OUTPUT_DIR = Path("/content/drive/MyDrive/uncle/experiment_exports")
INCLUDE_CHECKPOINTS = False
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(f"Saved run folder not found: {DRIVE_ROOT}")
print("Source run folder:", DRIVE_ROOT)
print("Include checkpoint weights:", INCLUDE_CHECKPOINTS)


## EXPORT-01: Build the ZIP

The export copies the original files without changing their contents. `README.md` explains the bundle, `RUN_TIMELINES.md` lays out saved requests, and `manifest.json` lists copied files, checksums, and exclusions. The ZIP is written to a temporary name first, then renamed when complete.


In [ ]:
from datetime import datetime, timezone
from scripts.export_experiment_bundle import export_bundle

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
bundle_path = OUTPUT_DIR / f"experiment_evidence_{stamp}.zip"
manifest = export_bundle(
    REPO, DRIVE_ROOT, bundle_path, include_checkpoints=INCLUDE_CHECKPOINTS
)
print("Saved ZIP:", bundle_path)
print("Repository files:", manifest["counts"]["repository_files"])
print("Saved run files:", manifest["counts"]["drive_files"])
print("Excluded files:", len(manifest["excluded"]))
print("Known locations:", manifest["known_drive_paths"])


## FINISH-01: Check the ZIP and flush Drive

Send the ZIP file to someone who needs the evidence. They can read the included `README.md` first, then inspect original files and verify them against `manifest.json`. The archive's older records are transcriptions, and the manifest tells them which Drive files were present at export time.


In [ ]:
import json
from zipfile import ZipFile

with ZipFile(bundle_path) as bundle:
    assert bundle.testzip() is None
    saved_manifest = json.loads(bundle.read("manifest.json"))
    assert saved_manifest == manifest
    assert "README.md" in bundle.namelist()
print("ZIP checked:", bundle_path)
drive.flush_and_unmount(timeout_ms=600000)
print("Drive flushed. CPU export complete.")
